In [6]:
!git pull https://github.com/Bamboohoccode/TempoRun_hf

Cloning into 'TempoRun_hf'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 157 (delta 73), reused 157 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 65.79 KiB | 647.00 KiB/s, done.
Resolving deltas: 100% (73/73), done.


In [3]:
!pip install transformers
!pip install accelerate

In [12]:
!cd ~/Another_jupyter_env/TempoRun/TempoRun_hf/TempoRun2026_Baseline && pip install -r requirements.txt

  Using cached open_clip_torch-3.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached pillow-12.3.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached torchvision-0.28.0-cp314-cp314-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached timm-1.0.28-py3-none-any.whl.metadata (40 kB)
Using cached open_clip_torch-3.3.0-py3-none-any.whl (1.5 MB)
Using cached pillow-12.3.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)
Using cached opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (61.2 MB)
Using cached timm-1.0.28-py3-none-any.whl (2.6 MB)
Using cached ftfy-6.3.1-py3-none-any.whl (44 kB)
Using cached torchvision-0.28.0-cp314-cp314-manylinux_2_28_x86_64.whl (7.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [open_clip_torch]━━ 4/6 [timm]v-pyt

In [1]:
import requests
import torch
from PIL import Image

from transformers import AutoModel, AutoProcessor


model = AutoModel.from_pretrained("google/siglip2-base-patch16-224", device_map="auto", attn_implementation="sdpa")
processor = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)
candidate_labels = ["a Pallas cat", "a lion", "a Siberian tiger"]

# follows the pipeline prompt template to get same results
texts = [f'This is a photo of {label}.' for label in candidate_labels]

# IMPORTANT: we pass `padding=max_length` and `max_length=64` since the model was trained with this
inputs = processor(text=texts, images=image, padding="max_length", max_length=64, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits_per_image = outputs.logits_per_image
probs = torch.sigmoid(logits_per_image)
print(f"{probs[0][0]:.1%} that image 0 is '{candidate_labels[0]}'")

/home/ngkhtrinh/Another_jupyter_env/jupyter_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 408/408 [00:00<00:00, 16596.77it/s]


1.1% that image 0 is 'a Pallas cat'


In [8]:
print(f"{probs[0][2]:.1%} that image 0 is '{candidate_labels[2]}'")

0.1% that image 0 is 'a Siberian tiger'
